# Gemma4NPC-it - Flagship NPC Model

This notebook trains the flagship `Gemma4NPC-it` model on the combined `RolePlay-NPC-v2` dataset (PIPPA + synthetic 24-turn NPC conversations).


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
import math

model, tokenizer = FastModel.from_pretrained(
    model_name = "google/gemma-4-12B-it",
    max_seq_length = 4096,
    load_in_4bit = True,
)

model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="data/final/RolePlay-NPC-v2_train.jsonl", split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat, num_proc=4)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

class NaNDetectionCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            if math.isnan(logs["loss"]) or math.isinf(logs["loss"]):
                control.should_training_stop = True

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        output_dir = "outputs/Gemma4NPC-it",
        dataset_text_field = "text",
        max_seq_length = 4096,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        num_train_epochs = 1,
        learning_rate = 2e-5,
        lr_scheduler_type = "cosine",
        warmup_steps = 1000,
        max_grad_norm = 0.4,
        optim = "adamw_8bit",
        bf16 = True,
        logging_steps = 10,
        save_steps = 500,
    ),
)

trainer = train_on_responses_only(
    trainer, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n",
)
trainer.add_callback(NaNDetectionCallback())

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained_merged("outputs/Gemma4NPC-it/merged_float16", tokenizer, save_method="merged_16bit")